# 03 · Audio: voice verification (Member 3)

This notebook turns each member's voice recordings into a model that can confirm
a transaction. It produces:

| Output | Used by |
|---|---|
| `data/audio_features.csv` | assignment deliverable |
| `models/voice_model.pkl` | `scripts/voice_model.py` → `verify_voice()` in the app |
| `report/figures/audio_*.png` | report, voice section |

**Before running:** put the recordings in `data/audio/<firstname>/` named
`yes_approve.wav` and `confirm_transaction.wav` (extra takes: `yes_approve_2.wav`, …),
and the non-member's clip(s) in `data/unauthorized/`. Then choose **Run All**.

Feature extraction lives in `scripts/voice_model.py`, so the app processes a new
clip in exactly the same way as the training clips.

In [ ]:
import subprocess
import sys
import warnings
from pathlib import Path

import joblib
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, f1_score,
                             log_loss)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*n_fft.*too large.*")
warnings.filterwarnings("ignore", message=".*PySoundFile failed.*")
warnings.filterwarnings("ignore", message=".*audioread.*")

# Find the project root: the folder that contains scripts/voice_model.py
ROOT = Path.cwd().resolve()
while not (ROOT / "scripts" / "voice_model.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Open this notebook from inside the project folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
import voice_model as vm

AUDIO_DIR = ROOT / "data" / "audio"
UNAUTHORIZED_DIR = ROOT / "data" / "unauthorized"
FEATURES_CSV = ROOT / "data" / "audio_features.csv"
MODEL_PATH = ROOT / "models" / "voice_model.pkl"
FIG_DIR = ROOT / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

SR = vm.SAMPLE_RATE
SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "savefig.bbox": "tight"})

def rel(path):
    """Path relative to the project root, for tidy printing."""
    return Path(path).resolve().relative_to(ROOT).as_posix()

print("Project root:", ROOT)

## 1. Load and check the recordings

Every file is checked before it is used. A file is flagged if:

| Check | Rule | Fix |
|---|---|---|
| Too short | less than 0.5 s of speech | re-record, say the full phrase |
| Too long | file longer than 10 s | trim the silence |
| Too quiet | speech below −40 dBFS | record closer to the microphone |
| Clipping | more than 0.1 % of samples at full scale | record further away or lower the input gain |
| Unexpected name | not `yes_approve` / `confirm_transaction` (+ optional `_2`, `_3`) | rename the file |
| Unreadable | the file can't be decoded | export as `.wav` |

In [ ]:
MIN_SPEECH_SECONDS = 0.5
MAX_FILE_SECONDS = 10.0
QUIET_DBFS = -40.0
CLIP_LEVEL, MAX_CLIPPED = 0.99, 0.001

def check_recording(path, check_name=True):
    """Measure one file at its original sample rate and list any problems."""
    info = {"file": rel(path)}
    try:
        y, sr = librosa.load(path, sr=None, mono=True)
    except Exception as exc:
        return {**info, "issues": f"unreadable ({type(exc).__name__})"}
    speech, _ = librosa.effects.trim(y, top_db=vm.TRIM_TOP_DB)
    speech_rms = np.sqrt(np.mean(speech ** 2)) if speech.size else 0.0
    info.update(
        sample_rate=sr,
        duration_s=round(len(y) / sr, 2),
        speech_s=round(len(speech) / sr, 2),
        loudness_dbfs=round(20 * np.log10(speech_rms + 1e-12), 1),
        clipped_pct=round(100 * np.mean(np.abs(y) >= CLIP_LEVEL), 3),
    )
    issues = []
    if info["speech_s"] < MIN_SPEECH_SECONDS:
        issues.append("too short")
    if info["duration_s"] > MAX_FILE_SECONDS:
        issues.append("too long")
    if info["loudness_dbfs"] < QUIET_DBFS:
        issues.append("too quiet")
    if info["clipped_pct"] > 100 * MAX_CLIPPED:
        issues.append("clipping")
    if check_name and vm.parse_recording_name(path)[0] not in vm.PHRASES:
        issues.append("unexpected name")
    info["issues"] = ", ".join(issues)
    return info

recordings = pd.DataFrame(vm.find_recordings(AUDIO_DIR))
if recordings.empty:
    raise FileNotFoundError(
        f"No recordings in {rel(AUDIO_DIR)}. Add data/audio/<firstname>/yes_approve.wav etc."
    )
unauthorized = pd.DataFrame(vm.find_recordings(UNAUTHORIZED_DIR))

quality = pd.DataFrame(
    [{"member": r.member, **check_recording(r.path)} for r in recordings.itertuples()]
    + [{"member": "(unauthorized)", **check_recording(r.path, check_name=False)}
       for r in unauthorized.itertuples()]
)
display(quality)

flagged = quality[quality["issues"] != ""]
if flagged.empty:
    print("All recordings passed the checks.")
else:
    print("Re-record or fix these files:")
    for row in flagged.itertuples():
        print(f"  - {row.file}: {row.issues}")

In [ ]:
# Drop files that can't be read, then check that every member said both phrases.
unreadable = set(quality.loc[quality["issues"].str.startswith("unreadable"), "file"])
recordings = recordings[~recordings["path"].map(rel).isin(unreadable)].reset_index(drop=True)
unauthorized = (unauthorized[~unauthorized["path"].map(rel).isin(unreadable)].reset_index(drop=True)
                if not unauthorized.empty else unauthorized)
recordings["recording_id"] = recordings["member"] + "/" + recordings["path"].map(lambda p: p.name)

coverage = (recordings.groupby(["member", "phrase"]).size()
            .unstack(fill_value=0)
            .reindex(columns=list(vm.PHRASES), fill_value=0))
display(coverage)

MEMBERS = sorted(recordings["member"].unique())
for member in MEMBERS:
    missing = [p for p in vm.PHRASES if coverage.loc[member, p] == 0]
    if missing:
        print(f"WARNING: {member} has no recording of: {', '.join(missing)}")
    if (recordings["member"] == member).sum() < 2:
        print(f"WARNING: {member} has only one recording, so it can't be tested on an unseen clip.")
if len(MEMBERS) < 2:
    raise ValueError("The voice model needs recordings from at least two members.")
if unauthorized.empty:
    print("WARNING: no clip in data/unauthorized/, so the approval threshold can't be tuned.")
ONE_TAKE = bool((coverage.reindex(columns=list(vm.PHRASES)) <= 1).all().all())
if ONE_TAKE:
    print("NOTE: everyone has one take per phrase. Section 6 will then test whether the model\n"
          "      recognises a voice saying the *other* phrase, which is harder than the app's job.\n"
          "      A second take of each phrase (yes_approve_2.wav, ...) gives a fairer test.")

print(f"\n{len(recordings)} recordings from {len(MEMBERS)} members: {', '.join(MEMBERS)}")
print(f"{len(unauthorized)} unauthorized clip(s)")

## 2. Waveforms and spectrograms

Each clip is resampled to 16 kHz, trimmed of leading and trailing silence and
peak-normalised (the same processing the app applies), then plotted. The
**waveform** shows loudness over time; the **mel spectrogram** shows which
frequencies carry energy over time (brighter = louder).

In [ ]:
clips = {r.recording_id: vm.load_audio(r.path) for r in recordings.itertuples()}
unauthorized_clips = {f"unauthorized/{r.path.name}": vm.load_audio(r.path)
                      for r in unauthorized.itertuples()}

# First take of each phrase for each member
examples = (recordings.sort_values("take")
            .drop_duplicates(["member", "phrase"])
            .set_index(["member", "phrase"])["recording_id"])
found = set(recordings["phrase"])
phrases = [p for p in vm.PHRASES if p in found] + sorted(found - set(vm.PHRASES))

def example_grid(plot_fn, title, filename):
    fig, axes = plt.subplots(len(MEMBERS), len(phrases),
                             figsize=(6 * len(phrases), 2.4 * len(MEMBERS)),
                             squeeze=False, sharey="row")
    for i, member in enumerate(MEMBERS):
        for j, phrase in enumerate(phrases):
            ax = axes[i, j]
            rid = examples.get((member, phrase))
            if rid is None:
                ax.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue
            plot_fn(clips[rid], ax)
            ax.set_title(f"{member} · {phrase.replace('_', ' ')}", fontsize=10)
            ax.label_outer()
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename)
    plt.show()

def plot_wave(y, ax):
    librosa.display.waveshow(y, sr=SR, ax=ax, color="steelblue")
    ax.set_ylim(-1, 1)

def plot_mel(y, ax):
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=64), ref=np.max)
    librosa.display.specshow(mel, sr=SR, x_axis="time", y_axis="mel", ax=ax, cmap="magma")

example_grid(plot_wave, "Waveforms per member and phrase", "audio_waveforms.png")
example_grid(plot_mel, "Mel spectrograms per member and phrase (dB)", "audio_spectrograms.png")

In [ ]:
# Listen to each example
for (member, phrase), rid in examples.items():
    print(f"{member} · {phrase}")
    display(Audio(clips[rid], rate=SR))
for name, y in unauthorized_clips.items():
    print(name)
    display(Audio(y, rate=SR))

## 3. What the plots show

*Write this in your own words after looking at the plots and listening to the clips.
Some prompts:*

- *Waveforms: how do the members differ in length, loudness and pauses between words?
  Do the two phrases have a recognisable shape?*
- *Spectrograms: where is each voice's energy concentrated? Lower voices show
  harmonic bands packed closer together. Which voices look most alike?*
- *Quality: is there background noise (a haze across all frequencies), clipping
  (flat tops on the waveform) or long silence?*
- *What does this suggest about how easy each member will be to tell apart?*

**Interpretation:** _Write here._

## 4. Augmentation

With only a few recordings per person the model would memorise them. Each
recording gets five altered copies so it learns the voice rather than one take:

| Augmentation | Setting | Simulates |
|---|---|---|
| `pitch_up` / `pitch_down` | ±1.5 semitones | a slightly higher or lower voice (tired, excited) |
| `stretch_fast` / `stretch_slow` | 1.15× / 0.85× speed | speaking faster or slower |
| `noise` | pink background noise at 20 dB SNR | a room with a fan, traffic or distant chatter |

Pitch shifts are kept small on purpose: a large shift would change who the voice
sounds like. Each copy is peak-normalised again, like a real clip.

In [ ]:
def checked(fn, y, **kwargs):
    """Run a librosa effect and check the result is valid audio.

    librosa 1.0's phase vocoder prints a harmless "invalid value encountered in
    cast" warning on some clips; it is silenced here and the output checked instead.
    """
    with warnings.catch_warnings(), np.errstate(invalid="ignore"):
        warnings.filterwarnings("ignore", message="invalid value encountered in cast")
        out = fn(y, **kwargs)
    assert np.isfinite(out).all(), f"{fn.__name__} produced invalid samples"
    return out

def pitch_shift(y, n_steps):
    return checked(librosa.effects.pitch_shift, y, sr=SR, n_steps=n_steps)

def time_stretch(y, rate):
    return checked(librosa.effects.time_stretch, y, rate=rate)

def add_noise(y, snr_db):
    """Add pink noise (more energy at low frequencies, like real room noise)."""
    spectrum = np.fft.rfft(rng.normal(size=y.size))
    freqs = np.fft.rfftfreq(y.size, d=1 / SR)
    spectrum[0] = 0
    spectrum[1:] /= np.sqrt(freqs[1:])
    noise = np.fft.irfft(spectrum, n=y.size)
    target_power = np.mean(y ** 2) / (10 ** (snr_db / 10))
    return y + noise * np.sqrt(target_power / np.mean(noise ** 2))

AUGMENTATIONS = {
    "pitch_up": lambda y: pitch_shift(y, +1.5),
    "pitch_down": lambda y: pitch_shift(y, -1.5),
    "stretch_fast": lambda y: time_stretch(y, 1.15),
    "stretch_slow": lambda y: time_stretch(y, 0.85),
    "noise": lambda y: add_noise(y, snr_db=20),
}

def normalise(y):
    peak = np.max(np.abs(y))
    return (0.95 * y / peak if peak > 0 else y).astype(np.float32)

def all_versions(y):
    """The original clip plus one copy per augmentation."""
    return {"original": y, **{name: normalise(fn(y)) for name, fn in AUGMENTATIONS.items()}}

augmented = {rid: all_versions(y) for rid, y in clips.items()}
augmented_unauthorized = {rid: all_versions(y) for rid, y in unauthorized_clips.items()}
n_versions = len(AUGMENTATIONS) + 1
print(f"{len(augmented)} recordings x {n_versions} versions = {len(augmented) * n_versions} member clips")
print(f"{len(augmented_unauthorized)} unauthorized recordings x {n_versions} versions "
      f"= {len(augmented_unauthorized) * n_versions} unauthorized clips")

In [ ]:
# Show every version of one recording
example_id = examples.iloc[0]
versions = augmented[example_id]
fig, axes = plt.subplots(len(versions), 2, figsize=(12, 1.9 * len(versions)), sharex="col")
for i, (name, y) in enumerate(versions.items()):
    plot_wave(y, axes[i, 0])
    plot_mel(y, axes[i, 1])
    axes[i, 0].set_ylabel(name, rotation=0, ha="right", va="center", fontsize=10)
    axes[i, 1].set_ylabel("")
    for ax in axes[i]:
        if i < len(versions) - 1:
            ax.set_xlabel("")
axes[0, 0].set_title("Waveform")
axes[0, 1].set_title("Mel spectrogram")
fig.suptitle(f"Augmentations of {example_id}", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "audio_augmentation_example.png")
plt.show()

for name, y in versions.items():
    print(name)
    display(Audio(y, rate=SR))

## 5. Feature extraction → `audio_features.csv`

Each clip (original or augmented) becomes one row of 48 numbers, computed by
`vm.extract_features` in `scripts/voice_model.py`:

| Feature | Columns | What it captures |
|---|---|---|
| MFCC 1–20, mean and std | 40 | the shape of the vocal tract; the classic speaker-recognition feature |
| Spectral roll-off, mean and std | 2 | frequency below which 85 % of the energy lies (brightness) |
| RMS energy, mean and std | 2 | loudness and how much it varies |
| Zero-crossing rate, mean | 1 | how noisy or breathy the voice is |
| Spectral centroid, mean | 1 | the "centre of mass" of the spectrum |
| Pitch, median (Hz) and spread (semitones) | 2 | how high the voice is and how much it moves; changes little between phrases |

The CSV also stores which member, phrase, recording and augmentation each row
came from. Unauthorized clips are included with `authorized = False`; they are
never used for training.

In [ ]:
rows = []
meta = recordings.set_index("recording_id")
for rid, versions in augmented.items():
    for aug, y in versions.items():
        rows.append({"member": meta.loc[rid, "member"], "phrase": meta.loc[rid, "phrase"],
                     "take": meta.loc[rid, "take"], "recording_id": rid, "augmentation": aug,
                     "authorized": True, **vm.extract_features(y, SR)})
for rid, versions in augmented_unauthorized.items():
    for aug, y in versions.items():
        rows.append({"member": "unauthorized", "phrase": vm.parse_recording_name(rid)[0],
                     "take": vm.parse_recording_name(rid)[1], "recording_id": rid,
                     "augmentation": aug, "authorized": False, **vm.extract_features(y, SR)})

features = pd.DataFrame(rows)
features.to_csv(FEATURES_CSV, index=False)
print(f"Saved {rel(FEATURES_CSV)}: {features.shape[0]} rows x {features.shape[1]} columns "
      f"({len(vm.FEATURE_COLUMNS)} features)")
display(features.groupby(["member", "augmentation"]).size().unstack(fill_value=0))
display(features.head())

In [ ]:
# Do the members form separate groups? Project the features onto 2 dimensions.
scaled = StandardScaler().fit_transform(features[vm.FEATURE_COLUMNS])
coords = PCA(n_components=2, random_state=SEED).fit_transform(scaled)

fig, ax = plt.subplots(figsize=(7, 5.5))
colors = plt.cm.tab10(np.arange(len(MEMBERS)))
for color, member in zip(colors, MEMBERS):
    for is_orig, marker, size in [(False, "o", 25), (True, "*", 160)]:
        mask = ((features["member"] == member) & (features["augmentation"].eq("original") == is_orig)).to_numpy()
        ax.scatter(coords[mask, 0], coords[mask, 1], color=color, marker=marker, s=size,
                   alpha=0.9 if is_orig else 0.45, edgecolor="k" if is_orig else "none",
                   label=member if is_orig else None)
mask = (~features["authorized"]).to_numpy()
if mask.any():
    ax.scatter(coords[mask, 0], coords[mask, 1], color="black", marker="x", s=45, label="unauthorized")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("Audio features (PCA): stars = original recordings, dots = augmented")
ax.legend()
fig.savefig(FIG_DIR / "audio_features_pca.png")
plt.show()

## 6. Train and evaluate the voice model

**Why not a normal train/test split?** The augmented copies of a recording are
almost identical to it. If a copy lands in training and the original in testing,
the score is inflated. So every recording and its copies stay together.

**Leave-one-recording-out:** for each recording, train on all the other
recordings (with their copies) and test on the held-out recording and its copies.
Every recording gets tested once by a model that never heard it, which uses the
small dataset fully.

**How many takes matters.** With one take per phrase, holding out someone's
"Yes, approve" leaves only their "Confirm transaction" to learn from, so the test
becomes "recognise this voice saying a phrase you never heard from them". The
app's real job is easier: a new take of a phrase the model *has* heard. With two
or more takes per phrase, the test matches that job.

Two models are compared: **Random Forest** and **Logistic Regression** (on
standardised features). The one with the higher macro F1 (then lower log loss)
is kept and retrained on all recordings.

In [ ]:
members_df = features[features["authorized"]].reset_index(drop=True)
X = members_df[vm.FEATURE_COLUMNS]
y = members_df["member"].to_numpy()
groups = members_df["recording_id"].to_numpy()
X_unauth = features.loc[~features["authorized"], vm.FEATURE_COLUMNS]
CLASSES = MEMBERS

MODELS = {
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            random_state=SEED),
    "Logistic Regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(max_iter=5000, class_weight="balanced")),
}

def full_proba(model, X_):
    """predict_proba with a column for every member, even one missing from a training fold."""
    out = np.zeros((len(X_), len(CLASSES)))
    proba = model.predict_proba(X_)
    for j, cls in enumerate(model.classes_):
        out[:, CLASSES.index(cls)] = proba[:, j]
    return out

is_original = members_df["augmentation"].eq("original").to_numpy()
results, oof = {}, {}
for name, model in MODELS.items():
    proba = np.zeros((len(X), len(CLASSES)))
    unauth_scores = []
    for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
        fold_model = clone(model).fit(X.iloc[train_idx], y[train_idx])
        proba[test_idx] = full_proba(fold_model, X.iloc[test_idx])
        if len(X_unauth):
            unauth_scores.append(full_proba(fold_model, X_unauth).max(axis=1))
    pred = np.array(CLASSES)[proba.argmax(axis=1)]
    results[name] = {
        "accuracy": accuracy_score(y, pred),
        "f1_macro": f1_score(y, pred, average="macro", labels=CLASSES, zero_division=0),
        "log_loss": log_loss(y, proba, labels=CLASSES),
        "accuracy_originals_only": accuracy_score(y[is_original], pred[is_original]),
    }
    oof[name] = {"proba": proba, "pred": pred,
                 "unauth_scores": np.concatenate(unauth_scores) if unauth_scores else np.array([])}

comparison = pd.DataFrame(results).T.round(3)
display(comparison)

BEST = min(results, key=lambda n: (-results[n]["f1_macro"], results[n]["log_loss"]))
print(f"Chosen model: {BEST}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
comparison[["accuracy", "f1_macro"]].plot.bar(ax=axes[0], rot=0, color=["steelblue", "darkorange"])
axes[0].set_ylim(0, 1.25)
axes[0].set_title("Accuracy and macro F1 (higher is better)")
axes[0].legend(["accuracy", "macro F1"], loc="upper center", ncol=2)
comparison["log_loss"].plot.bar(ax=axes[1], rot=0, color="gray")
axes[1].set_title("Log loss (lower is better)")
for ax in axes:
    for bar in ax.patches:
        ax.annotate(f"{bar.get_height():.2f}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha="center", va="bottom", fontsize=9)
fig.suptitle("Voice model comparison (leave-one-recording-out)", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "audio_model_comparison.png")
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y, oof[BEST]["pred"], labels=CLASSES, ax=ax,
                                        cmap="Blues", colorbar=False)
ax.set_title(f"{BEST}: held-out predictions")
fig.savefig(FIG_DIR / "audio_confusion_matrix.png")
plt.show()

# Which recordings does the model struggle with?
per_recording = (members_df.assign(correct=oof[BEST]["pred"] == y)
                 .groupby("recording_id")["correct"].mean().rename("share_correct").to_frame())
display(per_recording.sort_values("share_correct"))

## 7. Approval threshold

The model always names *someone*, even for a stranger. To reject strangers, a
clip is approved only if the claimed member is the top prediction **and** the
model's confidence reaches a threshold.

The threshold is tuned with two sets of held-out scores from Section 6:

- **Genuine:** each member's held-out clips, scored by a model that never heard them.
- **Unauthorized:** the non-member's clips, scored by the same models. Their
  score is the confidence of whichever member they most resemble (the worst case).

The chosen threshold maximises *genuine accepted − unauthorized accepted*
(searching 0.50–0.95). If there's a tie, it takes the middle of the tied range.
If no threshold does better than accepting nobody, the scores can't separate the
two groups yet, so the default of 0.60 is kept and a warning is printed.

In [ ]:
true_idx = np.array([CLASSES.index(c) for c in y])
genuine_scores = oof[BEST]["proba"][np.arange(len(y)), true_idx]
genuine_correct = oof[BEST]["pred"] == y
unauth_scores = oof[BEST]["unauth_scores"]

candidates = np.round(np.arange(0.50, 0.951, 0.01), 2)
sweep = pd.DataFrame({
    "threshold": candidates,
    "genuine_accepted": [np.mean(genuine_correct & (genuine_scores >= t)) for t in candidates],
    "unauthorized_accepted": [np.mean(unauth_scores >= t) if len(unauth_scores) else np.nan
                              for t in candidates],
})

sweep["margin"] = sweep["genuine_accepted"] - sweep["unauthorized_accepted"]
if not len(unauth_scores):
    THRESHOLD = vm.DEFAULT_THRESHOLD
    print(f"No unauthorized clips: using the default threshold {THRESHOLD}.")
elif sweep["margin"].max() <= 0:
    THRESHOLD = vm.DEFAULT_THRESHOLD
    print(f"WARNING: no threshold separates members from the unauthorized voice on held-out\n"
          f"clips, so the default {THRESHOLD} is used. More takes per member should fix this.")
else:
    tied = sweep[np.isclose(sweep["margin"], sweep["margin"].max())]
    THRESHOLD = float(tied["threshold"].iloc[len(tied) // 2])

def pct(value):
    return "n/a" if pd.isna(value) else f"{value:.1%}"

at = sweep.loc[np.isclose(sweep["threshold"], THRESHOLD)].iloc[0]
print(f"Threshold: {THRESHOLD:.2f}")
print(f"  Genuine clips accepted:      {pct(at['genuine_accepted'])}")
print(f"  Unauthorized clips accepted: {pct(at['unauthorized_accepted'])}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
bins = np.linspace(0, 1, 21)
axes[0].hist(genuine_scores[genuine_correct], bins=bins, alpha=0.7, label="genuine (correct member)")
if len(unauth_scores):
    axes[0].hist(unauth_scores, bins=bins, alpha=0.7, color="crimson", label="unauthorized")
axes[0].axvline(THRESHOLD, color="k", ls="--", label=f"threshold {THRESHOLD:.2f}")
axes[0].set_xlabel("model confidence")
axes[0].set_ylabel("clips")
axes[0].set_title("Confidence on held-out clips")
axes[0].legend()
axes[1].plot(sweep["threshold"], sweep["genuine_accepted"], label="genuine accepted")
if len(unauth_scores):
    axes[1].plot(sweep["threshold"], sweep["unauthorized_accepted"], color="crimson",
                 label="unauthorized accepted")
axes[1].axvline(THRESHOLD, color="k", ls="--")
axes[1].set_xlabel("threshold")
axes[1].set_ylabel("share of clips")
axes[1].set_ylim(-0.02, 1.02)
axes[1].set_title("Effect of the threshold")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "audio_threshold.png")
plt.show()

In [ ]:
# Retrain the chosen model on every member recording and save it with the threshold.
final_model = clone(MODELS[BEST]).fit(X, y)
assert list(final_model.classes_) == CLASSES

bundle = {
    "model": final_model,
    "model_name": BEST,
    "classes": CLASSES,
    "feature_columns": vm.FEATURE_COLUMNS,
    "threshold": THRESHOLD,
    "sample_rate": SR,
    "metrics": {k: round(float(v), 4) for k, v in results[BEST].items()},
    "n_recordings": int(len(recordings)),
    "augmentations": list(AUGMENTATIONS),
}
joblib.dump(bundle, MODEL_PATH)
vm._MODEL_CACHE.clear()  # make verify_voice() load the new file
print(f"Saved {rel(MODEL_PATH)} ({BEST}, threshold {THRESHOLD:.2f})")

## 8. Test `verify_voice`

This is the function the app calls. Each member's first recording should be
**approved** for that member and **denied** when it claims to be someone else;
every unauthorized clip should be denied whoever it claims to be.

These member clips were part of training, so they only prove the plumbing works.
The real test is a **new** recording: save it anywhere outside `data/audio/` and run
`python scripts/voice_model.py <file> <name>` from the project folder.

In [ ]:
checks = []
first_takes = recordings.sort_values("take").drop_duplicates("member").set_index("member")["path"]
for member in CLASSES:
    path = first_takes[member]
    for claim in CLASSES:
        approved, confidence = vm.verify_voice(path, claim, MODEL_PATH)
        checks.append({"clip": rel(path), "claimed": claim,
                       "expected": "approve" if claim == member else "deny",
                       "result": "approve" if approved else "deny",
                       "confidence": round(confidence, 3)})
for path in unauthorized["path"] if not unauthorized.empty else []:
    for claim in CLASSES:
        approved, confidence = vm.verify_voice(path, claim, MODEL_PATH)
        checks.append({"clip": rel(path), "claimed": claim, "expected": "deny",
                       "result": "approve" if approved else "deny",
                       "confidence": round(confidence, 3)})

checks = pd.DataFrame(checks)
checks["correct"] = checks["expected"] == checks["result"]
display(checks)
print(f"{checks['correct'].sum()}/{len(checks)} checks behaved as expected")

In [ ]:
# The same check from a terminal, as the app or a teammate would run it
demo_member = CLASSES[0]
demo_clip = first_takes[demo_member]
command = [sys.executable, str(ROOT / "scripts" / "voice_model.py"), rel(demo_clip), demo_member]
print("$ python scripts/voice_model.py", rel(demo_clip), demo_member)
result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
print(result.stdout or result.stderr)

## 9. Summary for the report

Copy these numbers into the README results table and the report. With only a few
recordings per member, one misclassified clip moves the scores a lot, so say how
many recordings the numbers are based on.

In [ ]:
per_member = recordings.groupby("member").size()
figures = sorted(p.name for p in FIG_DIR.glob("audio_*.png"))
lines = [
    "VOICE MODEL SUMMARY",
    "===================",
    f"Members enrolled:        {len(CLASSES)} ({', '.join(CLASSES)})",
    f"Recordings:              {len(recordings)} "
    f"({', '.join(f'{m}: {n}' for m, n in per_member.items())})",
    f"Unauthorized clips:      {len(unauthorized)}",
    f"Augmentations:           {len(AUGMENTATIONS)} per recording ({', '.join(AUGMENTATIONS)})",
    f"Feature rows:            {len(features)} ({len(members_df)} member, "
    f"{len(features) - len(members_df)} unauthorized), {len(vm.FEATURE_COLUMNS)} features each",
    "Evaluation:              leave-one-recording-out (copies stay with their recording)",
    "",
    comparison.to_string(),
    "",
    f"Chosen model:            {BEST}",
    f"Approval threshold:      {THRESHOLD:.2f}",
    f"  genuine accepted:      {pct(at['genuine_accepted'])}",
    f"  unauthorized accepted: {pct(at['unauthorized_accepted'])}",
    f"verify_voice checks:     {checks['correct'].sum()}/{len(checks)} as expected",
    "",
    "Figures in report/figures/: " + ", ".join(figures),
]
print("\n".join(lines))